create spark session
 

In [0]:
# Create a Spark session
# SparkSession is the entry point to Spark functionality
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName('my spark')  # Set a name for the application
    .getOrCreate()         # Get an existing session or create a new one
    
)


In [0]:
spark

In [0]:
# Define sample employee data as a list of lists.
# Each inner list represents one employee record with the following columns:
#   [employee_id, department_id, name, age, gender, salary, hire_date]
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

# Define the schema for the employee DataFrame using a DDL-formatted string.
# All columns are stored as strings (age, salary, etc. can be cast later as needed).
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


In [0]:
# Create a DataFrame from the sample employee data and schema defined earlier.
# spark.createDataFrame takes the raw data and applies the DDL-formatted schema string.
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

# Display the DataFrame contents in a rich interactive table format in the notebook output.
display(emp)

In [0]:
#show data(action)
emp.show()

In [0]:
emp_final=emp.where("salary>50000")
emp_final.show()

In [0]:
# for schema 
emp.printSchema()
emp.schema


In [0]:
# Import PySpark data types needed to define a schema programmatically.
#   StructType  – represents a row schema (a collection of fields)
#   StructField – represents a single column inside the StructType
#   IntegerType – maps to the Spark 'int' data type
#   StringType  – maps to the Spark 'string' data type
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

# A schema can be defined using a DDL-formatted string (similar to SQL column definitions).
# This is a quick, readable way to specify column names and their data types.
schema_string='name string, age int'

# Alternatively, a schema can be built programmatically using StructType and StructField.
# Each StructField takes: column name, data type, and whether the column is nullable (True = allows nulls).
schema_park=StructType([
    StructField("name",StringType(),True),   # 'name' column of type string, nullable
    StructField("age",IntegerType(),True),   # 'age' column of type int, nullable
])

In [0]:

# Columns and expression
from pyspark.sql.functions import col, expr

emp["salary"]   

In [0]:
# print the employee_id, name, age, and salary columns from the emp DataFrame
# Import column-expression helpers: col() selects a column by name, expr() parses a SQL expression string
from pyspark.sql.functions import col, expr
# emp.show()  # (optional) show the full emp DataFrame for reference
# Select only the needed columns using three different approaches:
#   col('employee_id')  – select via the col() function
#   expr('name')        – select via a SQL expression string
#   emp.salary / emp.age – select via DataFrame attribute (Column object)
emp_filter = emp.select(col('employee_id'), expr('name'), emp.salary, emp.age)
# Trigger the action and display the selected columns
emp_filter.show()

In [0]:
# Rename employee_id as emp_id and cast age from string to int.
# Using expr() for column aliasing and type casting, and DataFrame attribute for name and salary.
emp_casted=emp.select(expr("employee_id as emp_id"),emp.name,expr("cast(age as int) as age"),emp.salary)
emp_casted.show()
#

In [0]:
# filter age > 30
emp_final=emp_casted.select(emp.age,emp.emp_id,emp.name,emp.salary).where(emp.age>30)
emp_final.show()

In [0]:
# Show the original employee DataFrame (all columns, all rows)
emp.show()

# Show the filtered DataFrame with selected columns: employee_id, name, salary, age
emp_filter.show()

# Show the final DataFrame after filtering employees with age > 30
emp_final.show()

# Show the casted DataFrame with emp_id alias and age cast to int
emp_casted.show()

In [0]:
emp_age_double=emp.selectExpr("employee_id","name","cast(salary as double) as salary","age")
emp_age_double.show()
 # or 
from pyspark.sql.functions import col,cast
emp_casted_age=emp.select('employee_id','name','age',col("salary").cast("double"))
emp_casted_age.printSchema()

In [0]:
emp_casted_age.show()


In [0]:
# add tax col as salry*0.2
emp_tax=emp_casted_age.withColumn('tax',col('salary')*0.2)
emp_tax.show()

In [0]:
# Add static (literal) value columns to the emp_tax DataFrame.
# lit() creates a Column with a constant value — useful for adding fixed/default columns.
from pyspark.sql.functions import lit

# withColumn() adds a new column (or replaces an existing one of the same name).
# Here we add two new columns with constant values:
#   'columnOne' – integer literal 500 (applied to every row)
#   'columnTwo' – string literal 'two' (applied to every row)
# Multiple withColumn calls can be chained; each extends the DataFrame schema.
emp_new_col = emp_tax.withColumn('columnOne', lit(500)).withColumn('columnTwo', lit('two'))

# Trigger the action to display the updated DataFrame with the new columns
emp_new_col.show()


In [0]:
# Rename an existing column using withColumnRenamed().
# withColumnRenamed() takes two arguments:
#   - the current column name (employee_id)
#   - the new column name (emp_id)
# It returns a new DataFrame with the specified column renamed; the original DataFrame is unchanged.

emp_1 = emp_new_col.withColumnRenamed('employee_id', 'emp_id')

# Trigger the action to display the DataFrame with the renamed column
emp_1.show()


In [0]:
# ---------------------------------------------------------------------------
# Renaming a column to a name that contains a space.
#
# withColumnRenamed('columnTwo', 'column two') changes the column name from
# 'columnTwo' to 'column two' (note the space in the new name).
#
# IMPORTANT: Column names with spaces are generally discouraged because they
# can cause issues downstream. For example, if a downstream consumer expects
# a column named 'columnTwo' (without a space), renaming it to 'column two'
# will break references that use dot-notation or SQL-style column access.
# Always verify that downstream data consumers can handle space-containing
# column names before applying such a rename.
# ---------------------------------------------------------------------------

# Rename 'columnTwo' to 'column two' (with a space) in the emp_1 DataFrame.
emp_2 = emp_1.withColumnRenamed('columnTwo', 'column two')

# Trigger the action to display the DataFrame with the renamed column.
emp_2.show()

In [0]:
# remove colums 
emp_dropped=emp_new_col.drop('columnTwo')
emp_dropped.show()


In [0]:
# filter data 
emp_filtered=emp_dropped.where('tax>10000')
emp_filtered.show()




In [0]:
# limit 
emp_limited=emp_filtered.limit(5)
emp_limited.show()

In [0]:
# add n number of columns at once using withColumns()
# withColumns() takes a dict of {column_name: expression}
from pyspark.sql.functions import col, lit

columns = {
    "tax": col('salary') * 0.2,
    "columnOne": lit(500),
    "columnTwo": lit('two')
}

emp_finaldf = emp.withColumns(columns)
emp_finaldf.show()

In [0]:
emp.show()

In [0]:
# ---------------------------------------------------------------------------
# CASE WHEN logic using PySpark's when() / otherwise() functions.
#
# when(condition, value) acts like a CASE WHEN clause in SQL:
#   - Each .when() checks a condition and returns a value if true.
#   - .otherwise() acts like the SQL ELSE clause — the fallback value
#     when none of the preceding conditions match.
#
# Here we create a new column 'new_gender' that shortens the
# 'gender' values: 'Male' -> 'M', 'Female' -> 'F', anything else -> NULL.
# ---------------------------------------------------------------------------

from pyspark.sql.functions import when

emp_gender_fixed = emp.withColumn(
    "new_gender",
    when(col('gender') == "Male", "M")
    .when(col('gender') == 'Female', "F")
    .otherwise(None)
)
emp_gender_fixed.show()

In [0]:
# ---------------------------------------------------------------------------
# String replacement using regexp_replace().
#
# regexp_replace(column, pattern, replacement) finds all occurrences of a
# pattern (here the literal character 'J') in the specified column ('name')
# and replaces them with the given replacement (here 'Z').
#
# withColumn('new_name', ...) adds a new column called 'new_name' to the
# emp_gender_fixed DataFrame, containing the result of the replacement.
# The original 'name' column is left unchanged.
#
# Note: regexp_replace is case-sensitive by default, so only uppercase 'J'
# is replaced; lowercase 'j' would remain as-is.
# ---------------------------------------------------------------------------

from pyspark.sql.functions import regexp_replace

emp_name_fixed = emp_gender_fixed.withColumn('new_name', regexp_replace('name', 'J', 'Z'))
emp_name_fixed.show()

In [0]:
# ---------------------------------------------------------------------------
# Convert the 'hire_date' column from string to a proper Date type.
#
# First, print the schema of the emp DataFrame to confirm that 'hire_date'
# is currently stored as a string.
# ---------------------------------------------------------------------------
emp.printSchema()

# ---------------------------------------------------------------------------
# Use to_date() to parse the 'hire_date' string column into a Spark DateType.
#   to_date(col, format) – converts a string column to a date using the
#   specified format pattern (here 'yyyy-MM-dd').
#
# withColumn() overwrites the existing 'hire_date' column in emp_name_fixed
# with the converted date values, producing a new DataFrame: emp_date_fix.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import to_date, col

emp_date_fix = emp_name_fixed.withColumn('hire_date', to_date(col('hire_date'), 'yyyy-MM-dd'))

# Display the updated data and verify the schema change (hire_date should
# now appear as 'date' instead of 'string').
emp_date_fix.show()
emp_date_fix.printSchema()


In [0]:
# ---------------------------------------------------------------------------
# Add the current date and timestamp as new columns to the emp_date_fix DataFrame.
#
# current_date()     – returns the current date (a DateType column, no time portion).
# current_timestamp() – returns the current timestamp (a TimestampType column,
#                       includes both date and time down to seconds/fractional).
#
# withColumn() adds a new column to the DataFrame. Multiple withColumn() calls
# can be chained to add several columns at once.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import current_date, current_timestamp

# Add 'date_now' (today's date) and 'timestamp_now' (exact current timestamp).
emp_dated = emp_date_fix.withColumn('date_now', current_date()).withColumn('timestamp_now', current_timestamp())

# Display the DataFrame with the new columns.
# By default, .show() truncates long column values to 20 characters for readability.
emp_dated.show()

# Passing truncate=False disables truncation so the full (untruncated) value of
# every column is shown — useful for inspecting long strings, full timestamps,
# or wide columns that would otherwise be cut off.
emp_dated.show(truncate=False)

In [0]:
# ---------------------------------------------------------------------------
# Handling NULL values in a DataFrame.
#
# Approach 1 (NOT recommended): Drop all rows containing any NULL value.
#   na.drop() returns a new DataFrame with rows removed if *any* column is NULL.
#   This is a brute-force approach — it discards entire rows and loses data,
#   even when only a single column has a missing value. Use this only when
#   you are certain that NULL rows are not needed.
# ---------------------------------------------------------------------------
emp_1 = emp_dated.na.drop()

# ---------------------------------------------------------------------------
# Approach 2 (recommended): Replace NULLs with a meaningful default value.
#
#   coalesce(col, lit) returns the first non-NULL value among its arguments:
#     - If 'new_gender' is not NULL, keep its original value.
#     - If 'new_gender' is NULL, substitute the literal 'O' (Other).
#
#   This preserves all rows while ensuring no NULL remains in the target column.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import coalesce, lit

emp_null_df = emp_dated.withColumn('new_gender', coalesce(col('new_gender'), lit('O')))
emp_null_df.show()

In [0]:
# drop name and gender col and rename new_name as name and new_gender as gender
emp_final_df=emp_null_df.drop('name','gender').withColumnRenamed('new_name',"name").withColumnRenamed('new_gender',"gender")

emp_final_df.show()

In [0]:
# Write emp_final_df to a CSV file in DBFS
emp_final_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/FileStore/emp_final_df_csv")

# Read it back to verify the saved CSV
emp_csv_readback = spark.read \
    .option("header", True) \
    .csv("/FileStore/emp_final_df_csv")

emp_csv_readback.show()

In [0]:
emp_data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],]

emp_data_2=[["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]]


# Define the schema for the employee DataFrame using a DDL-formatted string.
# All columns are stored as strings (age, salary, etc. can be cast later as needed).
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [0]:
emp_data_1 = spark.createDataFrame(data=emp_data_1, schema=emp_schema)
emp_data_2 = spark.createDataFrame(data=emp_data_2, schema=emp_schema)

emp_data_1.show()
emp_data_2.show()

In [0]:
# ---------------------------------------------------------------------------
# UNION vs UNION ALL in PySpark
#
# Before combining two DataFrames, both must have the SAME schema:
#   - Same number of columns
#   - Columns must be in the same order
#   - Column data types must match
# This is why we print the schemas of both DataFrames first — to verify
# they are compatible before attempting a union.
# ---------------------------------------------------------------------------
emp_data_1.printSchema()
emp_data_2.printSchema()

# ---------------------------------------------------------------------------
# union() — combines two DataFrames by stacking their rows.
#   - By default, union() performs a UNION ALL (it does NOT remove duplicates).
#   - It matches columns strictly by POSITION (not by column name), so the
#     column order in both DataFrames must be identical.
#   - If you need to remove duplicates, chain .dropDuplicates() after union(),
#     e.g. emp_data_1.union(emp_data_2).dropDuplicates().
# ---------------------------------------------------------------------------
emp = emp_data_1.union(emp_data_2)

# ---------------------------------------------------------------------------
# unionAll() — is an alias for union().
#   - In PySpark, union() and unionAll() behave the SAME way (both keep
#     duplicates). unionAll() is the older name retained for backward
#     compatibility; union() is the preferred method in modern code.
#   - Neither union() nor unionAll() deduplicates rows.
# ---------------------------------------------------------------------------
emp_unionall = emp_data_1.unionAll(emp_data_2)

# Display both results to confirm they are identical.
emp.show()
emp.printSchema()
emp_unionall.show()


In [0]:
# ---------------------------------------------------------------------------
# Sorting a DataFrame by a column in ascending or descending order.
#
# sort() returns a new DataFrame sorted by the given column expression.
#   - col('salary').desc  – sorts in descending order (highest salary first).
#   - col('salary').asc   – sorts in ascending order (lowest salary first).
#
# .desc and .asc are properties on a Column object, so there is no need to
# import the standalone desc()/asc() functions. Only col is imported.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import col

# Sort employees by salary descending (highest first)
emp_sorted_desc = emp.sort(col('salary').desc())

# Sort employees by salary ascending (lowest first)
emp_sorted_asc = emp.sort(col('salary').asc())

# Trigger actions to display both sorted DataFrames
emp_sorted_desc.show()
emp_sorted_asc.show()

In [0]:
# ---------------------------------------------------------------------------
# Aggregation examples using groupBy() and agg().
#
# groupBy('column') groups rows by the specified column(s), and agg()
# applies one or more aggregate functions (count, sum, avg, min, max) to
# each group. Each aggregate result can be given a readable alias name.
# ---------------------------------------------------------------------------

# Import common aggregate functions from pyspark.sql.functions.
from pyspark.sql.functions import avg, count, max, min, sum

# ---------------------------------------------------------------------------
# 1) Count the number of employees in each department.
#
# groupBy('department_id') groups rows by department.
# count('employee_id') counts non-null employee_id values per group.
# alias('employee_count') names the resulting column for clarity.
# ---------------------------------------------------------------------------
emp_count = emp_sorted_desc.groupBy("department_id").agg(count("employee_id").alias("employee_count"))
emp_count.show()

# ---------------------------------------------------------------------------
# 2) Total salary for each department.
#
# sum('salary') adds up all salary values within each department group.
# alias('Total_salary') names the resulting column.
# ---------------------------------------------------------------------------
emp_salary = emp_sorted_desc.groupBy('department_id').agg(sum('salary').alias('Total_salary'))
emp_salary.show()

# ---------------------------------------------------------------------------
# 3) Average salary for each department.
#
# avg('salary') computes the mean salary within each department group.
# alias('avg_salary') names the resulting column.
# ---------------------------------------------------------------------------
emp_avg_salary = emp_sorted_desc.groupBy('department_id').agg(avg('salary').alias('avg_salary'))
emp_avg_salary.show()

In [0]:
# ---------------------------------------------------------------------------
# Aggregation (groupBy) with a HAVING clause equivalent.
#
# In SQL, HAVING filters groups AFTER aggregation. In PySpark, the same
# effect is achieved by calling .where() (or .filter()) AFTER .agg(),
# because the aggregation has already produced the 'avg_salary' column.
#
# Here we:
#   1. groupBy('department_id')  – group rows by department
#   2. agg(avg('salary')...)     – compute average salary per department
#   3. where('avg_salary>50000') – keep only departments whose average
#                                 salary exceeds 50,000 (HAVING clause)
# ---------------------------------------------------------------------------
emp_salary_avg = (
    emp_sorted_desc
    .groupBy('department_id')
    .agg(avg('salary').alias('avg_salary'))
    .where('avg_salary > 50000')
)
emp_salary_avg.show()

In [0]:
# ---------------------------------------------------------------------------
# Union of two DataFrames whose columns are in a DIFFERENT order.
#
# emp_data_1 and emp_data_2 both share the same schema (same columns, same
# types), but the column ORDER differs. The regular union() / unionAll()
# methods match columns strictly by POSITION, which would misalign data if
# the column order doesn't match.
#
# To handle this safely we use unionByName(), which matches columns by NAME
# instead of by position — so the order of columns in each DataFrame does
# not matter as long as the column names are the same.
#
# Step 1: Reorder the columns of emp_data_2 using select() so they are in
#         a different sequence than emp_data_1. This demonstrates that
#         unionByName() can still align them correctly.
# ---------------------------------------------------------------------------
emp_data_2_other = emp_data_2.select(
    "employee_id", "salary", 'name', 'age', 'department_id', 'gender', 'hire_date'
)

# Print the schema and data of the reordered DataFrame to confirm the new
# column order before performing the union.
emp_data_2_other.printSchema()
emp_data_2_other.show()

# ---------------------------------------------------------------------------
# Step 2: Use unionByName() to combine emp_data_1 and emp_data_2_other.
#
# unionByName() stacks the rows of both DataFrames, matching each column by
# its name (not its position). This avoids data misalignment even though the
# two DataFrames have columns in a different order.
#
# Unlike union() (which matches by position), unionByName() is safer when
# you are not certain the column orders are identical.
# ---------------------------------------------------------------------------
emp_fixed = emp_data_1.unionByName(emp_data_2_other)

# Display the combined result and its schema to verify that the union was
# performed correctly and all rows from both DataFrames are present.
emp_fixed.show()
emp_fixed.printSchema()